# Recover information from epi-bit sequencing

This notebook adapts the read-count, methylation classification and comparison workflow in the original Hu analysis. Run it with either an ordered payload-and-seed CSV (PVS mode) or read-level 5mC probabilities and a write design (methylation mode). The final BCH decoder recovers any supported byte payload, including images and archives.

The default configuration recovers the bundled panda AVIF from its nanopore PVS data. For another experiment, use the original byte count recorded during encoding and set the output extension to match the encoded file.


In [ ]:
import csv
import hashlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.mixture import GaussianMixture

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "codec").is_dir() else cwd.parent
if not (REPO_ROOT / "codec" / "panda_decoding.py").is_file():
    raise FileNotFoundError("Start Jupyter from the repository root or data_analysis directory.")
sys.path.insert(0, str(REPO_ROOT / "codec"))
from bch_codes import CODEWORD_BITS, DATA_BITS
from panda_decoding import decode_file

MODE = "pvs"  # "pvs" or "methylation"
PVS_PATH = REPO_ROOT / "codec" / "pvs.csv"
ORIGINAL_SIZE_BYTES = 7882
OUTPUT_DIR = REPO_ROOT / "analysis_output"
OUTPUT_NAME = "recovered.avif"
EXPECTED_SHA256 = None  # Optional original file hash

# Methylation-mode inputs.
CALLS_PATH = Path("calls/per_read_5mc.tsv")
LAYOUT_PATH = Path("layout.csv")
ASSIGNMENTS_PATH = Path("read_assignments.csv")
THRESHOLDS_PATH = None  # Optional per-site calibration CSV
FIT_THRESHOLDS = True  # Estimate from the observed payloads if no calibration is supplied.
DEFAULT_THRESHOLD = 0.5
MIN_READS_PER_BIT = 1

# Optional: use dynamic clustering output instead of ASSIGNMENTS_PATH.
CLUSTER_ITER_DIR = None  # e.g. Path("clustered/iter_0")
CLUSTER_MAP_PATH = None  # CSV with cluster_index,row

# Optional: compare with the PVS generated when encoding the source.
EXPECTED_PVS = None


## Methylation-mode input tables

Use the per_read_5mc.tsv produced by the calling script. Its columns are read_id, chrm, pos and mod_log_prob, where pos is a zero-based coordinate and mod_log_prob is the natural log of the 5mC probability.

- layout.csv records the physical design. Use columns row, chrm, pos, bit_index, seed. Rows and bit indices are contiguous and zero-based. Each row represents one payload, in encoder order, and has one sparsification seed. A position must occur at most once within a row.
- read_assignments.csv contains read_id and row, one assignment per read. Use the assignment from barcode decoding. To use an iteration of dynamic clustering, set CLUSTER_ITER_DIR and supply a cluster_index,row table. The cluster index matches the integer filename written by dynamic_clustering.py.
- A calibration table, if supplied, has columns chrm, pos and threshold. Otherwise, the notebook estimates a two-component Gaussian mixture across payload rows at each site. It uses DEFAULT_THRESHOLD when a fit cannot resolve two groups. Use independently calibrated thresholds when available.
- If the original encoder PVS is available, set EXPECTED_PVS to compare against the expected payload bits. Ground truth is required for a confusion matrix and counts of errors before BCH correction.


In [ ]:
def read_columns(path, columns, sep=","):
    frame = pd.read_csv(path, sep=sep, dtype={"read_id": "string", "chrm": "string"})
    if missing := set(columns) - set(frame):
        raise ValueError(f"{path} is missing columns: {', '.join(sorted(missing))}")
    frame = frame[list(columns)]
    if frame.empty or frame.isna().any().any():
        raise ValueError(f"{path} must contain complete rows for the required columns.")
    return frame


def validate_integers(frame, columns, allow_negative=False):
    for column in columns:
        numeric = pd.to_numeric(frame[column], errors="raise")
        if not np.isfinite(numeric).all() or (numeric % 1 != 0).any():
            raise ValueError(f"{column} must contain integers.")
        if not allow_negative and (numeric < 0).any():
            raise ValueError(f"{column} must be nonnegative.")
        frame[column] = numeric.astype("int64")


def read_assignments():
    if CLUSTER_ITER_DIR is None and CLUSTER_MAP_PATH is None:
        frame = read_columns(ASSIGNMENTS_PATH, ("read_id", "row"))
    elif CLUSTER_ITER_DIR is None or CLUSTER_MAP_PATH is None:
        raise ValueError("Set CLUSTER_ITER_DIR and CLUSTER_MAP_PATH together.")
    else:
        mapping = read_columns(CLUSTER_MAP_PATH, ("cluster_index", "row"))
        validate_integers(mapping, ("cluster_index", "row"))
        if mapping["cluster_index"].duplicated().any():
            raise ValueError("Each cluster index must be listed once.")
        groups = []
        for cluster, row in mapping.itertuples(index=False, name=None):
            path = Path(CLUSTER_ITER_DIR) / f"{cluster}.txt"
            groups.append(read_columns(path, ("read_id",)).assign(row=row))
        frame = pd.concat(groups, ignore_index=True)
    validate_integers(frame, ("row",))
    if frame["read_id"].duplicated().any():
        raise ValueError("Each read ID must be assigned to at most one payload row.")
    return frame


def load_payload_sites():
    layout = read_columns(LAYOUT_PATH, ("row", "chrm", "pos", "bit_index", "seed"))
    validate_integers(layout, ("row", "pos", "bit_index", "seed"))
    layout = layout.sort_values(["row", "bit_index"])
    if layout.duplicated(["row", "chrm", "pos"]).any():
        raise ValueError("A reference site must occur at most once per payload row.")
    if not np.array_equal(layout["row"].unique(), np.arange(layout["row"].nunique())):
        raise ValueError("Payload rows must be consecutive from zero.")
    groups = layout.groupby("row", sort=False)
    widths = groups.size()
    bit_indices = groups.cumcount()
    if widths.nunique() != 1 or not layout["bit_index"].eq(bit_indices).all():
        raise ValueError("Rows need equal widths and consecutive bit indices starting at zero.")
    if groups["seed"].nunique().ne(1).any() or layout["seed"].gt(2**32 - 1).any():
        raise ValueError("Each row needs one valid 32-bit sparsification seed.")

    assignments = read_assignments()
    if not assignments["row"].isin(layout["row"]).all():
        raise ValueError("Assignments contain a payload row outside the layout.")
    calls = read_columns(CALLS_PATH, ("read_id", "chrm", "pos", "mod_log_prob"), sep="\t")
    validate_integers(calls, ("pos",))
    log_prob = pd.to_numeric(calls.pop("mod_log_prob"), errors="raise")
    if (log_prob > 0).any() or log_prob.isna().any():
        raise ValueError("mod_log_prob must be the natural log of a probability in [0, 1].")
    calls["probability"] = np.exp(log_prob)
    calls = calls.merge(assignments, on="read_id", how="inner", validate="many_to_one")
    # Count each read once per site even if the source repeats a call.
    per_read = calls.groupby(["row", "chrm", "pos", "read_id"], as_index=False)["probability"].mean()
    site_values = per_read.groupby(["row", "chrm", "pos"], as_index=False).agg(
        probability=("probability", "median"),
        reads=("read_id", "size"),
    )
    return layout.merge(site_values, on=["row", "chrm", "pos"], how="left", validate="one_to_one"), assignments


## Classify each payload bit and assemble PVS rows

The median 5mC probability across reads assigned to each payload site is compared with the selected site threshold. Sites below MIN_READS_PER_BIT are reported in site_calls.csv. Recovery stops if a payload site lacks sufficient coverage.


In [ ]:
def site_threshold(values):
    x = values.dropna().to_numpy()
    if not FIT_THRESHOLDS or len(x) < 4 or len(np.unique(x)) < 2:
        return DEFAULT_THRESHOLD, "fallback"
    model = GaussianMixture(n_components=2, covariance_type="spherical", random_state=42)
    model.fit(x.reshape(-1, 1))
    low, high = np.sort(model.means_.ravel())
    if not model.converged_ or high - low < 0.05:
        return DEFAULT_THRESHOLD, "fallback"
    return (low + high) / 2, "mixture_midpoint"


def add_thresholds(sites):
    keys = ["chrm", "pos"]
    if THRESHOLDS_PATH is not None:
        thresholds = read_columns(THRESHOLDS_PATH, ("chrm", "pos", "threshold"))
        validate_integers(thresholds, ("pos",))
        thresholds["threshold"] = pd.to_numeric(thresholds["threshold"], errors="raise")
        thresholds["method"] = "calibration"
    else:
        records = []
        eligible = sites["probability"].where(sites["reads"].ge(MIN_READS_PER_BIT))
        for (chrm, pos), group in eligible.groupby([sites["chrm"], sites["pos"]], sort=False):
            value, method = site_threshold(group)
            records.append((chrm, pos, value, method))
        thresholds = pd.DataFrame(records, columns=[*keys, "threshold", "method"])
    result = sites.merge(thresholds, on=keys, how="left", validate="many_to_one")
    if not result["threshold"].between(0, 1).all():
        raise ValueError("Provide a threshold between zero and one for every payload site.")
    return result, thresholds


def infer_pvs():
    sites, assignments = load_payload_sites()
    sites, thresholds = add_thresholds(sites)
    thresholds.to_csv(OUTPUT_DIR / "thresholds.csv", index=False)
    counts = assignments.groupby("row").size().reindex(sites["row"].unique(), fill_value=0)
    counts.rename("assigned_reads").rename_axis("row").to_csv(OUTPUT_DIR / "read_counts.csv")
    coverage_ok = sites["reads"].ge(MIN_READS_PER_BIT)
    sites["bit"] = sites["probability"].gt(sites["threshold"]).astype("Int8").where(coverage_ok)
    sites.to_csv(OUTPUT_DIR / "site_calls.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
    ax.hist(thresholds["threshold"], bins=np.linspace(0, 1, 21), color="#28658a")
    ax.set(xlabel="5mC threshold", ylabel="Reference sites")
    fig.savefig(OUTPUT_DIR / "threshold_distribution.png", dpi=180)
    plt.show()
    display(counts.describe().rename("assigned_reads"))
    if not coverage_ok.all():
        raise ValueError(f"Insufficient coverage at {(~coverage_ok).sum()} payload sites. See site_calls.csv.")

    output_pvs = OUTPUT_DIR / "recovered_pvs.csv"
    with output_pvs.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        for _, group in sites.sort_values(["row", "bit_index"]).groupby("row", sort=False):
            writer.writerow(["".join(group["bit"].astype(str)), int(group["seed"].iat[0])])
    return output_pvs


if MODE not in {"pvs", "methylation"}:
    raise ValueError("MODE must be pvs or methylation.")
if not isinstance(ORIGINAL_SIZE_BYTES, int) or ORIGINAL_SIZE_BYTES <= 0:
    raise ValueError("ORIGINAL_SIZE_BYTES must be a positive integer.")
if MIN_READS_PER_BIT < 1 or not 0 <= DEFAULT_THRESHOLD <= 1:
    raise ValueError("Select a positive read count and a default threshold in [0, 1].")
if Path(OUTPUT_NAME).name != OUTPUT_NAME:
    raise ValueError("Set OUTPUT_NAME to a filename within OUTPUT_DIR.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pvs = Path(PVS_PATH) if MODE == "pvs" else infer_pvs()


## Compare with the original payload and decode

When EXPECTED_PVS is supplied, the comparison excludes template padding. It counts mismatches in the original BCH codeword order and indicates blocks exceeding the correction capacity of 24 bits. The decoder recovers the exact requested byte count. Use the original SHA-256 digest to verify content integrity when available.


In [ ]:
def read_payloads(path):
    with Path(path).open(newline="", encoding="utf-8") as source:
        rows = list(csv.reader(source))
    if not rows or any(len(row) < 2 or not row[0] or set(row[0]) - {"0", "1"} for row in rows):
        raise ValueError(f"Invalid payload data in {path}.")
    return [(row[0], int(row[1])) for row in rows]


def compare_payloads(observed_path, expected_path):
    observed, expected = map(read_payloads, (observed_path, expected_path))
    if len(observed) != len(expected) or any(
        len(a) != len(b) or seed_a != seed_b
        for (a, seed_a), (b, seed_b) in zip(observed, expected)
    ):
        raise ValueError("Observed and expected PVS must have identical row widths and seeds.")
    blocks = (ORIGINAL_SIZE_BYTES * 8 + DATA_BITS - 1) // DATA_BITS
    length = blocks * CODEWORD_BITS
    bits = [np.fromiter("".join(row[0] for row in payload)[:length], dtype=np.uint8)
            for payload in (observed, expected)]
    actual, truth = bits
    if len(actual) != length:
        raise ValueError("The PVS is too short for the stated file size.")
    confusion = pd.crosstab(truth, actual, rownames=["expected"], colnames=["observed"]).reindex(
        index=[0, 1], columns=[0, 1], fill_value=0
    )
    errors = (actual != truth).reshape(CODEWORD_BITS, blocks).sum(axis=0)
    block_report = pd.DataFrame({"block": np.arange(blocks), "errors": errors, "exceeds_limit": errors > 24})
    confusion.to_csv(OUTPUT_DIR / "confusion_matrix.csv")
    block_report.to_csv(OUTPUT_DIR / "bch_errors.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
    ax.plot(block_report["block"], errors, linewidth=1)
    ax.axhline(24, color="#a33b25", linestyle="--", label="BCH correction limit")
    ax.set(xlabel="BCH block", ylabel="Bit errors")
    ax.legend()
    fig.savefig(OUTPUT_DIR / "bch_errors.png", dpi=180)
    plt.show()
    display(confusion)
    print(f"Blocks exceeding the BCH correction limit: {(errors > 24).sum()} of {blocks}")


if EXPECTED_PVS is not None:
    compare_payloads(pvs, EXPECTED_PVS)
destination = OUTPUT_DIR / OUTPUT_NAME
decode_file(pvs, destination, ORIGINAL_SIZE_BYTES)
digest = hashlib.sha256(destination.read_bytes()).hexdigest()
print(f"SHA-256: {digest}")
if EXPECTED_SHA256 is not None and digest.lower() != EXPECTED_SHA256.strip().lower():
    raise ValueError("The recovered file does not match the original SHA-256 digest.")
